In [1]:
import math
from scipy.integrate import odeint
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import sys
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp 
from numpy import linalg as LA
import import_ipynb
import burgers_helper
from burgers_helper import *

In [2]:
# Function double_prod: Solving inner product of two Hermite polynomials
def double_prod(i: int, j: int):
    ''' Calculate double product for H_i, H_j

    Parameters
    ----------
    i: non-negative integer
    j: non-negative integer

    Returns
    -------
    i! if i = j
    0 if i != j
    '''
    if i != j:
        return 0
    else:
        return math.factorial(i)

In [3]:
# Function triple_prod: Solving inner product of three Hermite polynomials
def triple_prod(i: int, j: int, k: int):
    ''' Calculate triple product for H_i, H_j, H_k
    
    Parameters
    ----------
    i: non-negative integer
    j: non-negative integer
    k: non-negative integer

    Returns
    -------
    0 if (i+j+k)% 2 != 0 or s<max(i,j,k)
    i!j!k!/((s-i)!(s-j)!(s-k)!)
    '''
    s = (i+j+k)//2
    if (i+j+k)% 2 != 0 or s<max(i,j,k):
        return 0
    else:
        return (math.factorial(i) * math.factorial(j) * math.factorial(k)) / (
            math.factorial(s-i) *
            math.factorial(s-j) *
            math.factorial(s-k))

In [4]:
# Function approximate the continuous derivative of a function, using central difference
#or alternatively, forward difference

# Function xderiv_forward(y, delta_x)
    # y: Nx1 vector, including its boundary points
    # return a numerical approximation of dy/dx evaluated at each entry of y
def xderiv_forward(y, delta_x):
    N = len(y)
    dydx = np.zeros(N) #Initilize output vector
    
    # Forward difference for estimated values of y up to N-2 values of x
    for i in range (N-1):
        dydx[i] = (y[i+1] - y[i])/delta_x
    # Backward difference for estimated value of y at x_{N-1}
    dydx[N-1] = (y[N-1] - y[N-2])/delta_x
    return dydx

In [5]:
# Function xderiv_central(y, delta_x)
    # y: Nx1 vector, including its boundary points
    # return a numerical approximation of dy/dx evaluated at each entry of y
def xderiv_central(y, delta_x):
    N = len(y)
    dydx = np.zeros(N) #Initilize output vector
    
    # Forward difference at x_0
    dydx[0] = (y[1] - y[0]) / delta_x
    
    # Central difference for points x_1...x_{N-2}
    for i in range(1, N - 1):
        dydx[i] = (y[i + 1] - y[i - 1]) / (2 * delta_x)
        
    # Backward difference at last point x_{N-1}
    dydx[N - 1] = (y[N - 1] - y[N - 2]) / delta_x
    return dydx

In [6]:
# Function getallxderiv(u, delta_x)
    # Input: u: 2xN matrix
    # return du/dx (2xN matrix)
def getallxderiv(u, delta_x):
    u = np.asarray(u)
    num_row, N = u.shape # Get the dimensions of u
    dudx = np.zeros_like(u) # Initialize (M+1)xN matrix du/dx

    # Get derivatives for each row
    for i in range(num_row):
        dudx[i,:] = xderiv_central(u[i,:], delta_x) 
    return dudx

In [7]:
# Function get initial condition for the given PDE
def getinitcond(x, M, ic_type = 'cond_1'):
    ''' Function getinitcond: returns initial condition

    Parameters
    ----------
    x: array
        An array of length N. This can be understood as the gridpoints
    M: non-negative integer
        Truncation limit. For now, we are limiting our function to M = 1 and M = 2
    ic_type: string
        Type of initial condition we are working with. For now, there are 2 modes.
        cond_1: u(x,t=0,ξ) = ξ np.sin(x)
        cond_2: u(x,t=0,ξ) = ξ^2 np.sin(x)

    Returns
    -------
    u_init: (M+1)x N matrix
        Initial condition matrix
    '''
    N = len(x)
    u_init = np.zeros((M+1,N)) # Initialize initial condition matrix
    if ic_type == 'cond_1':
        if M >= 1:
            u_init[1,:] = np.sin(x)
    elif ic_type == 'cond_2':
        if M >= 1:
            u_init[0,:] = np.sin(x)
        if M >= 2:
            u_init[2,:] = np.sin(x)
    return u_init